# Exercise 1 — Notebook 1: Data Preparation

Before describing customer behaviour, computing risk indicators, or forecasting, 
we need a clean, well-defined dataset. This notebook:

1. Loads the raw monthly panel data (1,000 customers x 12 months)
2. Inspects and validates the data structure
3. Handles the mixed text/numeric `MonthsPastDue` column explicitly
4. Builds the derived columns every later notebook will reuse: 
   utilisation ratio, repayment ratio, payment compliance ratio, default flag
5. Saves a clean processed dataset for notebooks 2, 3, and 4 to load directly


## Step 1: Import libraries and load the data

We use `pandas` for data manipulation and `numpy` for numerical operations. We load the Excel file directly 
from the `data` subfolder so this notebook is fully self-contained.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
#here we load the raw data
df=pd.read_excel('data/Selection process for Risk Manager.xlsx', sheet_name='Input')
print(f"Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head(10)

Shape: 12000 rows, 9 columns


,CustomerID,ForDate,Age_FU,OpeningBalance,ClosingBalance,CreditLimit,MinPaymentAmount,Payments,MonthsPastDue
0,ID834,2022-10-01,2,20.05,95.72,3000,20.05,150.00,0
1,ID834,2022-11-01,3,95.72,245.72,3000,0.00,150.00,0
2,ID834,2022-12-01,4,245.72,225.11,3000,0.00,150.00,0
3,ID834,2023-01-01,5,225.11,177.24,3000,0.00,150.00,0
4,ID834,2023-02-01,6,177.24,187.64,3000,30.00,177.24,0
5,ID834,2023-03-01,7,187.64,2069.03,3000,30.00,187.64,0
6,ID834,2023-04-01,8,2069.03,1023.28,6000,62.07,2069.03,0
7,ID834,2023-05-01,9,1023.28,7.86,6000,30.70,1023.28,0
8,ID834,2023-06-01,10,7.86,203.39,6000,7.86,7.86,0
9,ID834,2023-07-01,11,203.39,0.85,6000,30.00,203.39,0


**Data description**

− CustomerID: unique client identifier.

− ForDate: indicates to which month and year a row is related (please ignore the day as it always refers to
the first day of the month).

− Age_FU: number of months since a client used their credit card for the first time, for instance:
    1 = a client has enacted a transaction with his card for the first time.
    10 = it represents month 10 of a client since his first card usage.

− OpeningBalance: outstanding loan balance at the beginning of the month.

− ClosingBalance: outstanding loan balance at the end of the month. (closing balance=OpeningBalance + (new spending this month) − (payments made this month))

− CreditLimit: assigned credit limit (maximum credit a customer can take advantage of).

− MinPaymentAmount: minimum amount a customer is obliged to pay on a given month.

− Payments: payments client made during the month.

− MonthsPastDue: the number of months a customer has missed paying the required minimum amount, 

---

## Step 2: Inspect the data structure

Before doing anything else, we confirm:
- The number of unique customers and the time span covered
- Data types of each column (dates should be dates, numbers should be numeric)
- Whether there are any missing values
- What values actually appear in `MonthsPastDue` — this column is known to mix 
  integers (0, 1, 2, 3) with the text value `"4+"`, which needs to be handled 
  before we can do any numeric work with it

In [2]:
print("Data types:")
print(df.dtypes)
print("")
print("Number of unique customers:", df['CustomerID'].nunique())
print("Date range:", df['ForDate'].min(), "to", df['ForDate'].max())
print("Number of unique months:", df['ForDate'].nunique())
print("Missing values per column:")
print(df.isnull().sum())
print("")
print("Unique values in MonthsPastDue:")
print(df['MonthsPastDue'].unique())
print("")
print("Value counts for MonthsPastDue:")
print(df['MonthsPastDue'].value_counts())

Data types:
CustomerID                  object
ForDate             datetime64[ns]
Age_FU                       int64
OpeningBalance             float64
ClosingBalance             float64
CreditLimit                  int64
MinPaymentAmount           float64
Payments                   float64
MonthsPastDue               object
dtype: object

Number of unique customers: 1000
Date range: 2022-10-01 00:00:00 to 2023-09-01 00:00:00
Number of unique months: 12
Missing values per column:
CustomerID          0
ForDate             0
Age_FU              0
OpeningBalance      0
ClosingBalance      0
CreditLimit         0
MinPaymentAmount    0
Payments            0
MonthsPastDue       0
dtype: int64

Unique values in MonthsPastDue:
[0 1 2 3 '4+']

Value counts for MonthsPastDue:
MonthsPastDue
0     10188
4+      735
1       702
2       228
3       147
Name: count, dtype: int64


---

## Step 3: Sort the data correctly

This is a critical and easy-to-miss step. The data is a **panel dataset** - 
multiple customers, each observed over multiple months. Any calculation that 
depends on time order (month-over-month changes, lagged values, sequences) 
**must** be calculated within each customer separately, should not be across customers.

We sort by `CustomerID` first, then `ForDate` within each customer. This 
ordering is the foundation that prevents a very common and serious bug: 
accidentally calculating a "previous month" value using a *different 
customer's* data just because their rows happen to be adjacent in the table.

In [3]:
df=df.sort_values(['CustomerID', 'ForDate']).reset_index(drop=True)
#below code should show the first 13 rows of the dataframe with only the columns CustomerID, ForDate, Age_FU, and MonthsPastDue
df.head(13)[['CustomerID', 'ForDate', 'Age_FU', 'MonthsPastDue']]

,CustomerID,ForDate,Age_FU,MonthsPastDue
0,ID1,2022-10-01,9,0
1,ID1,2022-11-01,10,0
2,ID1,2022-12-01,11,0
3,ID1,2023-01-01,12,0
4,ID1,2023-02-01,13,0
5,ID1,2023-03-01,14,0
6,ID1,2023-04-01,15,0
7,ID1,2023-05-01,16,0
8,ID1,2023-06-01,17,0
9,ID1,2023-07-01,18,0


---

## Step 4: Clean the MonthsPastDue column

We need **two versions** of this column, because it serves two different purposes:

1. **A clean categorical/ordinal version** — for grouping, segmenting, and 
   labelling (e.g. "this customer is in the '4+' bucket this month")
2. **A numeric version** — for any calculation that needs to treat it as a 
   number (e.g. checking if MonthsPastDue increased from last month)

We also build the **default flag** here: a customer-month is in default 
if `MonthsPastDue == "4+"`. This single flag will be used throughout every 
later notebook, so we build it once, correctly, right here.

In [4]:
#Numeric version: map "4+" to 4, everything else stays as its integer value
df['MonthsPastDue_num']=df['MonthsPastDue'].apply(lambda x:4 if x=='4+' else int(x))

#Clean categorical label, useful for grouping and chart axis labels
def mpd_label(x):
    if x== '4+':
        return '4+ (Default)'
    else:
        return str(int(x))

df['MonthsPastDue_label']=df['MonthsPastDue'].apply(mpd_label)
# Default flag: True if customer is in the "4+" bucket this month
df['is_default']=(df['MonthsPastDue']=='4+')
print(df['MonthsPastDue_num'].value_counts().sort_index())
print()
print(df['is_default'].value_counts())

MonthsPastDue_num
0    10188
1      702
2      228
3      147
4      735
Name: count, dtype: int64

is_default
False    11265
True       735
Name: count, dtype: int64


---

## Step 5: Build the `Credit utilisation ratio`

**Definition:** credit utilisation ratio=ClosingBalance/CreditLimit

This measures how much of the available credit a customer is actually using 
at the end of the month. It is one of the single most informative behavioural 
signals in a credit card portfolio:

- 0 to 10% - customer barely uses the card, lowest risk exposure
- 11% to 30% - considered safe zone, low risk
- 31% to 50% - Moderate Risk
- over 50% - High risk (customer is relying heavily 
  on available credit, which is a well-established early warning signal for 
  future default in credit card portfolios)

*The 30% Rule: Most financial institutions view keeping credit usage strictly under 30% as the minimum requirement for demonstrating responsible credit habits*

We compute this per customer-month, so we can later look at both its 
distribution across customers and its trend over time.

In [5]:
df['utilisation_ratio']=df['ClosingBalance']/df['CreditLimit']

#sanity check: utilisation should mostly be between 0 and 1, 
#though it CAN exceed 1 if a customer is over-limit (worth knowing if it happens)
print("Utilisation ratio summary statistics:")
print(df['utilisation_ratio'].describe())
print()
print("Number of customer-months with utilisation > 1 (over-limit):",(df['utilisation_ratio'] > 1).sum())

Utilisation ratio summary statistics:
count    12000.000000
mean         0.414045
std          0.422196
min          0.000000
25%          0.004000
50%          0.223242
75%          0.913225
max          1.607600
Name: utilisation_ratio, dtype: float64

Number of customer-months with utilisation > 1 (over-limit): 1450


**Interpretation:** The portfolio shows a right-skewed utilisation distribution — 
median utilisation is a comfortable 22%, but the mean (41%) is pulled up by a 
meaningful high-utilisation segment: the top quartile sits above 91%. Most 
notably, **1,450 customer-months (12.1% of the portfolio) show utilisation 
above 100%**, meaning the customer's balance exceeds their assigned credit 
limit — likely driven by accumulating interest/fees or a limit reduction 
while a balance is outstanding. This over-limit group is a clear candidate 
for closer risk monitoring and will be examined further in the indicators 
section.

Extras:

This dataset gives us utilisation on **one credit card per customer**. In a 
real bank setting, risk teams typically look at utilisation at two levels:

- **Aggregate utilisation** — total balance across *all* credit products a 
  customer holds, divided by their *total* available credit across all of 
  them combined
- **Individual product utilisation** — utilisation on each specific card or 
  facility separately

Both views matter, and they can disagree in an important way: a customer 
could look fine in aggregate (say, 30% utilisation across two cards combined) 
while one specific card is maxed out at 95%. Banks treat this as a real risk 
flag even when the aggregate number looks healthy, because maxing out a 
single facility is itself a behavioural warning sign — it can indicate 
liquidity stress on that specific line, regardless of what the customer's 
broader credit picture looks like.

---

## Step 6: Build the repayment ratio

**Definition:** Repayment ratio = Payments / OpeningBalance

This measures how much of the balance a customer owed at the start of the 
month they actually paid back during the month:

- Ratio close to 1 (or above) --> customer is paying off most/all of their 
  balance --> behaves like a **"Transactor"**
- Ratio close to 0 --> customer is paying very little relative to what they 
  owe --> behaves like a **"Revolver"**

**Important data handling decision:** when `OpeningBalance` is 0, this ratio 
is mathematically undefined (division by zero). Rather than letting this 
silently produce an error or an infinite value, we explicitly set these 
cases to `NaN` (missing) and we state clearly that we are doing this. This 
is exactly the kind of transparent, deliberate data-handling choice a risk 
analyst should always make explicit rather than hide.

In [6]:
# Use np.where to avoid division by zero; set ratio to NaN where OpeningBalance is 0
df['repayment_ratio']=np.where(df['OpeningBalance'] > 0,df['Payments']/df['OpeningBalance'],np.nan)

print("Number of customer-months with OpeningBalance = 0 (ratio set to NaN):",(df['OpeningBalance'] == 0).sum())
print()
print("Repayment ratio summary statistics (excluding NaN):")
print(df['repayment_ratio'].describe())

Number of customer-months with OpeningBalance = 0 (ratio set to NaN): 2056

Repayment ratio summary statistics (excluding NaN):
count     9944.000000
mean         7.640144
std        303.335256
min         -1.000000
25%          0.030001
50%          0.067934
75%          1.000000
max      28700.000000
Name: repayment_ratio, dtype: float64


**Interpretation:** The mean repayment ratio (7.64, i.e. 764%) is not 
representative and should be disregarded — it is heavily distorted by a small 
number of extreme outliers where `OpeningBalance` is very close to zero, 
which makes the ratio mathematically explode even for a normal-sized payment 
(max value of 28,700 confirms this). The **median (6.8%) and 25th percentile 
(3.0%) are the reliable summary figures** here, and they tell a clearer story: 
the typical customer repays only a small fraction of their opening balance 
each month — consistent with revolving credit behaviour rather than full 
balance pay-off. The 75th percentile of 1.0 shows a meaningful group of 
customers do pay their balance in full. The negative minimum value (-1) 
warrants a closer look, as it implies a negative payment in the raw data, 
likely a refund or credit adjustment. We will investigate the extreme values 
directly before deciding how to handle them in further analysis.

In [7]:
#Look at the rows with the most extreme repayment ratios
print("Top 10 highest repayment ratios:")
print(df.nlargest(10, 'repayment_ratio')[['CustomerID', 'ForDate', 'OpeningBalance', 'ClosingBalance', 'Payments', 'repayment_ratio']])
print()
print("Rows with negative repayment ratio:")
print(df[df['repayment_ratio'] < 0][['CustomerID', 'ForDate', 'OpeningBalance', 'ClosingBalance', 'Payments', 'repayment_ratio']])

Top 10 highest repayment ratios:
     CustomerID    ForDate  OpeningBalance  ClosingBalance  Payments  repayment_ratio
3783      ID382 2023-01-01            0.01            0.22    287.00     28700.000000
913       ID167 2022-11-01            0.57           26.34   3250.00      5701.754386
186       ID111 2023-04-01            0.12           11.27    482.00      4016.666667
9167      ID786 2023-09-01            0.17            0.50    665.00      3911.764706
2176      ID261 2023-02-01            0.29         2501.45    710.00      2448.275862
9430      ID805 2023-08-01            0.10          107.65    200.10      2001.000000
9420      ID805 2022-10-01            0.08            0.46    134.53      1681.625000
8150       ID71 2022-12-01            0.15          223.72    200.15      1334.333333
5798      ID533 2022-12-01            0.69          267.47    908.31      1316.391304
3784      ID382 2023-02-01            0.22            0.30    288.00      1309.090909

Rows with negative r

**Diagnostic findings:**

**Extreme high ratios** are confirmed to be a denominator artefact, not unusual 
customer behaviour. In every top-10 case, `OpeningBalance` is negligible 
(€0.01–€0.69) while `Payments` is a completely ordinary amount (€130–€3,250) — 
the customer simply owed almost nothing and made a normal payment, which 
mathematically produces a huge ratio. This confirms capping is the right fix: 
it removes the mathematical distortion without discarding real customer-months.

**Negative ratios** have a different and genuine cause: every negative-ratio 
row has a **negative `Payments` value**, consistent with a refund, fee 
reversal, or credit adjustment being recorded through the same field as 
payments. Notably, four customers (`ID328`, `ID632`, `ID755`, `ID978`) show 
this pattern repeatedly across multiple months — suggesting this is a 
recurring data feature for certain accounts rather than a one-off anomaly. 
We flag these separately rather than capping them, since capping addresses 
large positive outliers, not negative values arising from a different cause.

In [8]:
#Diagnosis code
# Re-cap repayment_ratio using the 99th percentile
repayment_cap = df['repayment_ratio'].quantile(0.99)

df['repayment_ratio_capped'] = df['repayment_ratio'].clip(upper=repayment_cap)
df['has_negative_payment'] = df['Payments'] < 0
print(f"Repayment ratio — values capped at {repayment_cap:.2f}: "
      f"{(df['repayment_ratio'] > repayment_cap).sum()} customer-months affected "
      f"({(df['repayment_ratio'] > repayment_cap).sum() / df['repayment_ratio'].notna().sum() * 100:.1f}%)")
print()
print("Repayment ratio (capped, corrected) summary statistics:")
print(df['repayment_ratio_capped'].describe())

Repayment ratio — values capped at 20.88: 100 customer-months affected (1.0%)

Repayment ratio (capped, corrected) summary statistics:
count    9944.000000
mean        0.721269
std         2.422507
min        -1.000000
25%         0.030001
50%         0.067934
75%         1.000000
max        20.875149
Name: repayment_ratio_capped, dtype: float64


**Interpretation**
the repayment ratio now capped at their own 99th percentile (20.88 and 55.32 respectively), 
each affecting exactly 1.0% of valid observations. In both cases, the 25th, 
50th, and 75th percentiles remain identical to the raw, uncapped data, 
confirming the true shape of normal customer behaviour was fully preserved 
and only the genuine statistical tail was adjusted. Negative values arising 
from negative `Payments` entries (likely refunds or adjustments, affecting 
the same 6 customers across both ratios) remain untouched and are tracked 
separately via the `has_negative_payment` flag for further investigation if 
needed, rather than being statistically corrected.

---

## Step 7: Build the minimum payment compliance ratio

**Definition:** Payment compliance ratio = Payments / MinPaymentAmount

This tells us whether a customer paid at least the minimum required amount:

- Ratio >= 1 -> customer met or exceeded their minimum payment obligation
- Ratio < 1 -> customer underpaid relative to what was required — an early 
  behavioural warning sign that often precedes a customer rolling into a 
  worse `MonthsPastDue` bucket next month

**Data handling decision:** when `MinPaymentAmount` is 0 (which happens when 
a customer owes nothing), this ratio is mathematically undefined, so we set 
it to `NaN` explicitly rather than letting it silently error or produce an 
infinite value.

We anticipate the same mathematical issue we found with the repayment ratio: 
whenever `MinPaymentAmount` is very small (close to zero) but `Payments` is a 
normal amount, the ratio will explode for purely mathematical reasons, not 
because the customer is behaving unusually. We will check for this directly 
rather than assuming it, and apply the same capping approach if confirmed.

In [9]:
df['payment_compliance_ratio']=np.where(
    df['MinPaymentAmount'] > 0,
    df['Payments'] / df['MinPaymentAmount'],np.nan)

print("Number of customer-months with MinPaymentAmount = 0 (ratio set to NaN):", 
      (df['MinPaymentAmount'] == 0).sum())
print()
print("Payment compliance ratio summary statistics (excluding NaN):")
print(df['payment_compliance_ratio'].describe())

Number of customer-months with MinPaymentAmount = 0 (ratio set to NaN): 3028

Payment compliance ratio summary statistics (excluding NaN):
count     8972.000000
mean        12.151987
std        315.327271
min         -0.124818
25%          1.000000
50%          1.429048
75%          6.424083
max      28700.000000
Name: payment_compliance_ratio, dtype: float64


In [10]:
# Top 10 highest payment compliance ratios
print("Top 10 highest payment compliance ratios:")
print(df.nlargest(10, 'payment_compliance_ratio')[
    ['CustomerID', 'ForDate', 'MinPaymentAmount', 'Payments', 'payment_compliance_ratio']
])
print()

# Any negative ratios
print("Rows with negative payment compliance ratio:")
print(df[df['payment_compliance_ratio'] < 0][
    ['CustomerID', 'ForDate', 'MinPaymentAmount', 'Payments', 'payment_compliance_ratio']
])

Top 10 highest payment compliance ratios:
     CustomerID    ForDate  MinPaymentAmount  Payments  payment_compliance_ratio
3783      ID382 2023-01-01              0.01    287.00              28700.000000
913       ID167 2022-11-01              0.57   3250.00               5701.754386
9167      ID786 2023-09-01              0.17    665.00               3911.764706
2176      ID261 2023-02-01              0.29    710.00               2448.275862
9430      ID805 2023-08-01              0.10    200.10               2001.000000
9420      ID805 2022-10-01              0.08    134.53               1681.625000
8150       ID71 2022-12-01              0.15    200.15               1334.333333
5798      ID533 2022-12-01              0.69    908.31               1316.391304
6956       ID62 2023-06-01              0.40    446.37               1115.925000
9161      ID786 2023-03-01              0.30    312.00               1040.000000

Rows with negative payment compliance ratio:
      CustomerID    F

**Diagnostic findings:** The same two patterns identified in the repayment 
ratio reappear here, confirming a shared root cause rather than two separate 
issues.

**Extreme high ratios** again come from a near-zero denominator: every top-10 
row shows a tiny `MinPaymentAmount` (€0.08–€0.69) against an ordinary 
`Payments` value. Notably, `ID382` (Jan 2023) again produces exactly 28,700 — 
the same customer-month seen in the repayment ratio outliers, confirming this 
is one underlying low-balance month producing extreme values across multiple 
derived ratios simultaneously, not two unrelated occurrences.

**Negative ratios** trace back to **the exact same five customers** found in 
the repayment ratio (`ID328`, `ID632`, `ID755`, `ID978`, `ID393`), on the same 
months, confirming this is a genuine, recurring account-level data feature 
(small negative payment entries, likely refunds or adjustments) rather than 
random noise. We apply the same treatment as before: cap the upper extreme 
values, and flag negative payments separately rather than capping them.

In [11]:
# Derive a data-driven cap: the 99th percentile of the valid ratio
compliance_cap = df['payment_compliance_ratio'].quantile(0.99)
print(f"99th percentile of payment_compliance_ratio: {compliance_cap:.2f}")
print()

df['payment_compliance_ratio_capped'] = df['payment_compliance_ratio'].clip(upper=compliance_cap)

print(f"Payment compliance ratio — values capped at {compliance_cap:.2f}: "
      f"{(df['payment_compliance_ratio'] > compliance_cap).sum()} customer-months affected "
      f"({(df['payment_compliance_ratio'] > compliance_cap).sum() / df['payment_compliance_ratio'].notna().sum() * 100:.1f}%)")
print()
print("Payment compliance ratio (capped) summary statistics:")
print(df['payment_compliance_ratio_capped'].describe())

99th percentile of payment_compliance_ratio: 55.32

Payment compliance ratio — values capped at 55.32: 90 customer-months affected (1.0%)

Payment compliance ratio (capped) summary statistics:
count    8972.000000
mean        6.139739
std        10.345464
min        -0.124818
25%         1.000000
50%         1.429048
75%         6.424083
max        55.322691
Name: payment_compliance_ratio_capped, dtype: float64


**Interpretation** 

Both the repayment ratio and the payment compliance ratio 
are now capped at their own 99th percentile (20.88 and 55.32 respectively), 
each affecting exactly 1.0% of valid observations. In both cases, the 25th, 
50th, and 75th percentiles remain identical to the raw, uncapped data, 
confirming the true shape of normal customer behaviour was fully preserved 
and only the genuine statistical tail was adjusted. Negative values arising 
from negative `Payments` entries (likely refunds or adjustments, affecting 
the same 6 customers across both ratios) remain untouched and are tracked 
separately via the `has_negative_payment` flag for further investigation if 
needed, rather than being statistically corrected.

In [12]:
print(f"Final shape: {df.shape[0]} rows, {df.shape[1]} columns")
print()
print("Final columns:")
print(df.columns.tolist())
print()
df.head(5)

Final shape: 12000 rows, 18 columns

Final columns:
['CustomerID', 'ForDate', 'Age_FU', 'OpeningBalance', 'ClosingBalance', 'CreditLimit', 'MinPaymentAmount', 'Payments', 'MonthsPastDue', 'MonthsPastDue_num', 'MonthsPastDue_label', 'is_default', 'utilisation_ratio', 'repayment_ratio', 'repayment_ratio_capped', 'has_negative_payment', 'payment_compliance_ratio', 'payment_compliance_ratio_capped']



,CustomerID,ForDate,Age_FU,OpeningBalance,ClosingBalance,CreditLimit,MinPaymentAmount,Payments,MonthsPastDue,MonthsPastDue_num,MonthsPastDue_label,is_default,utilisation_ratio,repayment_ratio,repayment_ratio_capped,has_negative_payment,payment_compliance_ratio,payment_compliance_ratio_capped
0,ID1,2022-10-01,9,158.01,233.13,6000,30.0,30.00,0,0,0,False,0.038855,0.189861,0.189861,False,1.000000,1.000000
1,ID1,2022-11-01,10,233.13,453.95,6000,30.0,793.50,0,0,0,False,0.075658,3.403680,3.403680,False,26.450000,26.450000
2,ID1,2022-12-01,11,453.95,57.00,6000,30.0,521.09,0,0,0,False,0.009500,1.147902,1.147902,False,17.369667,17.369667
3,ID1,2023-01-01,12,57.00,33.64,6000,30.0,380.00,0,0,0,False,0.005607,6.666667,6.666667,False,12.666667,12.666667
4,ID1,2023-02-01,13,33.64,15.07,6000,30.0,30.00,0,0,0,False,0.002512,0.891795,0.891795,False,1.000000,1.000000


## Step 8: Save the cleaned dataset

We save this processed dataframe so notebooks 2, 3, and 4 can load this 
clean version directly, without repeating any of the steps above.

In [ ]:
df.to_excel('data/cleaned_portfolio_data.xlsx', index=False)

print("Saved cleaned dataset to data/cleaned_portfolio_data.xlsx")
print(f"Shape saved: {df.shape}")